# Stage1 信用市场风险偏好来源预检

按冻结计划对 Futu OpenD 只读请求 2011-01-03 的 HYG/LQD 日线；本卡只核对来源能力，不构造因子或读取 SOXX 收益。


In [1]:
import json,hashlib,tarfile,tempfile,subprocess,shlex
from datetime import datetime,timezone
from pathlib import Path
from futu import OpenQuoteContext,KLType,AuType,RET_OK
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_credit_risk_appetite_source_preflight_20261023';plan=json.loads((card/'plan.json').read_text())
assert plan['frozen'] and not (card/'probe_run_state.json').exists()
raw_root=root/'data/raw/external/stage1_credit_risk_appetite_source_preflight_20261023';raw_root.mkdir(parents=True,exist_ok=True)
raw_path=raw_root/'futu_response.json'
assert not raw_path.exists()
state={'status':'started','market_requests':0,'order_calls':0,'model_fits':0,'native_accounts':0}
(card/'probe_run_state.json').write_text(json.dumps(state,indent=2)+'\n')
results=[]
ctx=OpenQuoteContext(host=plan['connection']['host'],port=plan['connection']['port'])
try:
 for symbol in plan['symbols_in_order']:
  state['market_requests']+=1;(card/'probe_run_state.json').write_text(json.dumps(state,indent=2)+'\n')
  requested=datetime.now(timezone.utc).isoformat()
  ret,data,page=ctx.request_history_kline(symbol,start=plan['start'],end=plan['end'],ktype=KLType.K_DAY,autype=AuType.NONE,max_count=plan['max_count'])
  received=datetime.now(timezone.utc).isoformat()
  rows=data.to_dict('records') if ret==RET_OK else []
  result={'symbol':symbol,'ret_code':int(ret),'error':None if ret==RET_OK else str(data),'requested_at_utc':requested,'received_at_utc':received,'page_req_key':str(page) if page is not None else None,'rows':rows}
  results.append(result)
  raw_path.write_text(json.dumps({'plan':plan,'sdk':'futu','results':results},ensure_ascii=False,sort_keys=True,default=str)+'\n')
  if ret!=RET_OK or len(rows)==0 or page is not None:break
finally:ctx.close()
assert state['market_requests']<=plan['budget']['market_requests_max']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
with tempfile.TemporaryDirectory(prefix='credit-futu-source-') as temp:
 archive=Path(temp)/'futu_response.tar.gz'
 with tarfile.open(archive,'w:gz') as out:out.add(raw_path,arcname='futu_response.json')
 digest=sha(archive);settings=_settings(root)
 receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='credit-futu-readback-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:
   run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   member=source.extractfile('futu_response.json');assert member and hashlib.sha256(member.read()).hexdigest()==sha(raw_path)
status='one_day_both_symbols_nonempty' if len(results)==2 and all(x['ret_code']==RET_OK and len(x['rows'])>0 and x['page_req_key'] is None for x in results) else 'needs_data_one_day_probe_failed'
report={'status':status,'market_requests':state['market_requests'],'symbols':[{'symbol':x['symbol'],'ret_code':x['ret_code'],'rows':len(x['rows']),'first_time_key':x['rows'][0].get('time_key') if x['rows'] else None,'page_req_key':x['page_req_key'],'error':x['error']} for x in results],'raw_response_sha256':sha(raw_path),'nas_archive_sha256':digest,'nas_archive_bytes':receipt['size_bytes'],'nas_remote_hash_verified':True,'nas_scoped_restore_verified':True,'model_fits':0,'native_accounts':0,'SOXX_outcome_reads':0}
(card/'probe_result.json').write_text(json.dumps(report,indent=2)+'\n')
state['status']='completed';state['nas_archive_sha256']=digest;(card/'probe_run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:report[k] for k in ('status','market_requests','symbols','nas_archive_sha256')})


{'status': 'needs_data_one_day_probe_failed', 'market_requests': 1, 'symbols': [{'symbol': 'US.HYG', 'ret_code': 0, 'rows': 0, 'first_time_key': None, 'page_req_key': None, 'error': None}], 'nas_archive_sha256': 'c224862db775525a83eff848bca31f9bcf6f9547f13ff32abbe7ffc408750988'}


## 中文结论

冻结单日只读请求中，Futu OpenD 对 US.HYG 的 2011-01-03 日线返回成功码但 0 行；按计划停止，未请求 LQD。原始响应已 NAS 归档并恢复核对，1 个市场请求、0 拟合、0 账户。只说明本次连接此日无交付，不外推全部历史或其他来源；信用风险偏好机制暂记 needs data，无 SOXX 超额证据。
